[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/14_agents_and_mcp.ipynb)

# 📓 Notebook 14 (fast track) — Agents, Tools & MCP

> **Module:** Agents, Tools & MCP · **Estimated time:** ~60 min · **Difficulty:** Intermediate

> **Fast track · the frontier.** You can now build AI workflows (NB 10), retrieval (NB 11), small agents (NB 12), and ship a project (NB 13). This finale condenses the full course's **Module 8** into one pass: production agent loops, **robust tools**, and the **Model Context Protocol (MCP)** — the open standard (from Anthropic) that lets one tool server work in Claude Desktop, Claude Code, or your own app.

Runs **100% offline** via deterministic stand-ins; the real `mcp`/`anthropic` SDKs are noted where they drop in. For the deep version (planning, reflection, multi-agent, full capstone) see [`../08_agents_tools_mcp/`](../08_agents_tools_mcp/).

## 🎯 Objectives

1. Run a **ReAct** agent loop with a step **budget**.
2. Give a tool a **JSON-Schema** and **validate** its arguments.
3. Build a tiny **MCP server + client** and call **tools / resources / prompts**.
4. Connect an agent that **discovers** its tools over MCP.

## ✅ Prerequisites

Fast-track NB 12 (tools & small agents), NB 4 (functions), JSON.

## 1. The agent loop (ReAct) with a budget

An **agent** lets the model decide the next step in a **bounded** loop; your code runs the tools. Offline we use a deterministic `policy()` stand-in (a real LLM replaces just this) — the loop never changes.

In [ ]:
import json, re

CSAT = {"chat": 4.1, "email": 3.6, "phone": 4.4, "social": 3.2}
def lookup_csat(channel): return {"channel": channel, "csat": CSAT.get(channel.lower())}
def calculator(expression):
    if not re.fullmatch(r"[0-9.+\-*/() ]+", expression): return {"error": "bad expr"}
    return {"result": round(eval(expression, {"__builtins__": {}}), 4)}
TOOLS = {"lookup_csat": lookup_csat, "calculator": calculator}

def policy(goal, scratch):
    """Stand-in for the model: emit {action,args} or {final}."""
    t = goal.lower(); used = {s.get("action") for s in scratch}
    m = re.search(r"\b(chat|email|phone|social)\b", t)
    if m and "lookup_csat" not in used:
        return {"action": "lookup_csat", "args": {"channel": m.group(1)}}
    if ("times" in t or "*" in t) and "calculator" not in used:
        csat = next((s["obs"]["csat"] for s in scratch if "csat" in s.get("obs", {})), None)
        n = re.search(r"(\d+(\.\d+)?)", t)
        if csat and n:
            return {"action": "calculator", "args": {"expression": f"{csat} * {n.group(1)}"}}
    facts = [s["obs"] for s in scratch]
    return {"final": facts[-1] if facts else "no answer"}

def agent(goal, tools, max_steps=5, verbose=True):
    scratch = []
    for i in range(1, max_steps + 1):
        d = policy(goal, scratch)
        if "final" in d:
            if verbose: print(f"[{i}] ✅ {d['final']}")
            return d["final"]
        name, args = d["action"], d["args"]
        obs = tools[name](**args) if name in tools else {"error": "no such tool"}
        scratch.append({"action": name, "obs": obs})
        if verbose: print(f"[{i}] 🔧 {name}({args}) -> {obs}")
    return "(budget exhausted)"

print(agent("What is chat's CSAT, times 3?", TOOLS))

**Why the budget matters:** an agent decides its own next step, so nothing inherently stops it. `max_steps` is the guardrail against an infinite loop. Drop it to 1 and a multi-step task halts cleanly instead of hanging:

In [ ]:
print(agent("phone csat times 5", TOOLS, max_steps=1, verbose=False))

## 2. Robust tools — schema + validation

A real tool is called by a model that may send wrong types or missing fields. Describe it with a **JSON-Schema** and **validate before running**, returning a clear error the model can recover from — never a raw traceback.

In [ ]:
SCHEMA = {"type": "object",
          "properties": {"channel": {"type": "string"},
                         "pct": {"type": "number", "minimum": 0, "maximum": 100}},
          "required": ["channel", "pct"]}

def validate(schema, args):
    problems = []
    props = schema["properties"]
    for r in schema.get("required", []):
        if r not in args: problems.append(f"missing '{r}'")
    for k, v in args.items():
        spec = props.get(k, {})
        if spec.get("type") == "number" and not isinstance(v, (int, float)):
            problems.append(f"'{k}' must be a number")
        if isinstance(v, (int, float)) and "maximum" in spec and v > spec["maximum"]:
            problems.append(f"'{k}' above {spec['maximum']}")
    return problems

print("good   :", validate(SCHEMA, {"channel": "chat", "pct": 50}))
print("bad    :", validate(SCHEMA, {"channel": "chat", "pct": 150}))
print("missing:", validate(SCHEMA, {"channel": "chat"}))

Wrap execution so every call returns the **same envelope** — success or failure:

In [ ]:
def safe_call(fn, schema, args):
    problems = validate(schema, args)
    if problems: return {"ok": False, "error": "; ".join(problems)}
    try: return {"ok": True, "result": fn(**args)}
    except Exception as e: return {"ok": False, "error": f"{type(e).__name__}: {e}"}

refund = lambda channel, pct: {"channel": channel, "refund_pct": pct}
print(safe_call(refund, SCHEMA, {"channel": "chat", "pct": 25}))
print(safe_call(refund, SCHEMA, {"channel": "chat", "pct": 999}))

## 3. MCP in miniature

The **Model Context Protocol** standardises how an app connects to tools, data, and prompts — *"a USB-C port for AI."* Three roles: **host** (the app + model), **client** (1:1 connector), **server** (your capabilities). A server exposes three primitives:

| Primitive | Controlled by | Like |
|---|---|---|
| **tools** | the model | a POST endpoint |
| **resources** | the application | a GET / a file |
| **prompts** | the user | a slash command |

Messages are **JSON-RPC 2.0**. Here's a working server + client, in process:

### 🔬 What actually happens — discover, then invoke

Before any tool runs, the **client** and **server** exchange two kinds of messages over a standard wire format. The whole protocol is just this back-and-forth — and seeing the literal request/response **dicts** is what makes MCP click:

```text
   ┌─────────── AI app ───────────┐                    ┌──── your code ────┐
   │   MCP CLIENT                 │                    │   MCP SERVER       │
   │                              │                    │                    │
   │  1. DISCOVER                 │  ── list_tools ──▶ │  exposes tools +   │
   │     "what tools exist?"      │ ◀── schemas ────── │  their schemas     │
   │                              │                    │                    │
   │  2. INVOKE                   │  ── call_tool ───▶ │  runs the function │
   │     call_tool(name, args)    │ ◀── result ─────── │  returns a result  │
   └──────────────────────────────┘                    └────────────────────┘
```

> 💡 The payoff: the client **never hard-codes** the tools. It asks the server *"what have you got?"* at runtime, reads the schemas, then calls by name. Write a server **once**, and any MCP client (Claude Desktop, Claude Code, your app) can discover and use it — that's the **N×M → N+M** glue collapse.

Below is the entire exchange as plain Python dicts — no network, no SDK, just the two messages crossing the wire in each direction.

In [ ]:
import json

# ── A MOCK MCP SERVER: exposes tools as {name: (schema, function)} ───────────
class MockMCPServer:
    """Holds tools + their JSON schemas. Responds to request dicts."""
    def __init__(self):
        self.tools = {}  # name -> {"schema":..., "fn":...}

    def add_tool(self, name, schema, fn):
        self.tools[name] = {"schema": schema, "fn": fn}

    def handle(self, request):
        """The server side of the wire: dict in -> dict out."""
        method = request["method"]
        if method == "list_tools":                     # 1. DISCOVER
            return {"tools": [{"name": n, "schema": t["schema"]}
                              for n, t in self.tools.items()]}
        if method == "call_tool":                      # 2. INVOKE
            p = request["params"]
            fn = self.tools[p["name"]]["fn"]
            return {"result": fn(**p["arguments"])}
        return {"error": f"unknown method {method!r}"}

# ── A MOCK MCP CLIENT: sends request dicts, reads response dicts ─────────────
class MockMCPClient:
    """Knows NOTHING about the tools until it asks the server."""
    def __init__(self, server):
        self.server = server

    def _send(self, request):
        print("   CLIENT --> SERVER:", json.dumps(request))
        response = self.server.handle(request)         # ← the "wire" (in-process)
        print("   SERVER --> CLIENT:", json.dumps(response))
        return response

    def list_tools(self):
        return self._send({"method": "list_tools"})["tools"]

    def call_tool(self, name, **arguments):
        req = {"method": "call_tool",
               "params": {"name": name, "arguments": arguments}}
        return self._send(req)["result"]

# ── Build a server with two tools (each with a tiny JSON schema) ─────────────
srv_demo = MockMCPServer()
srv_demo.add_tool(
    "get_csat",
    {"type": "object", "properties": {"channel": {"type": "string"}},
     "required": ["channel"]},
    lambda channel: {"channel": channel, "csat": {"chat": 4.1, "email": 3.6}.get(channel)},
)
srv_demo.add_tool(
    "add",
    {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
     "required": ["a", "b"]},
    lambda a, b: {"sum": a + b},
)

client_demo = MockMCPClient(srv_demo)

print("STEP 1 — DISCOVER (list_tools):")
tools = client_demo.list_tools()
print("   discovered:", [t["name"] for t in tools], "\n")

print("STEP 2 — INVOKE (call_tool):")
out = client_demo.call_tool("get_csat", channel="chat")
print("   -> returned:", out)

Read the printed dicts top to bottom — that **is** the protocol:

1. **`{"method": "list_tools"}`** → server replies with a list of `{name, schema}`. The client now *knows* `get_csat` and `add` exist and what arguments each takes — it discovered them, it didn't import them.
2. **`{"method": "call_tool", "params": {"name": ..., "arguments": {...}}}`** → server runs the matching function and replies with `{"result": ...}`.

Swap the in-process `self.server.handle(...)` call for a stdio pipe or an HTTP POST and you have a *real* MCP connection — the **message shapes don't change**. That's the entire trick: a fixed request/response format means any client can talk to any server.

> 🧠 **Mental model: MCP is USB-C for AI tools.** A USB-C port doesn't know or care what's plugged into it — a drive, a monitor, a charger — because the *plug shape* is standardized. MCP is that standard plug for tools: a client discovers and calls any server's tools through one fixed message format, so you build the server **once** and plug it into every client. The cells that follow just dress up this same `list_tools` / `call_tool` exchange in the real protocol's JSON-RPC envelope.

In [ ]:
def rpc(i, method, params=None): return {"jsonrpc": "2.0", "id": i, "method": method, "params": params or {}}

class MCPServer:
    def __init__(self): self.tools, self.resources, self.prompts = {}, {}, {}
    def handle(self, req):
        m, p, i = req["method"], req.get("params", {}), req["id"]
        if m == "tools/list":
            return {"jsonrpc": "2.0", "id": i, "result": {"tools": [
                {"name": n, "description": t["d"],
                 "inputSchema": t.get("schema", {"type": "object", "properties": {}})}
                for n, t in self.tools.items()]}}
        if m == "tools/call":
            try:                                  # a bad call must become a JSON-RPC
                out = self.tools[p["name"]]["fn"](**p.get("arguments", {}))
            except Exception as e:                # error — never a raw traceback
                return {"jsonrpc": "2.0", "id": i,
                        "error": {"code": -32602, "message": f"{type(e).__name__}: {e}"}}
            return {"jsonrpc": "2.0", "id": i,
                    "result": {"content": [{"type": "text", "text": json.dumps(out)}]}}
        if m == "resources/read":
            return {"jsonrpc": "2.0", "id": i,
                    "result": {"contents": [{"text": json.dumps(self.resources[p["uri"]]())}]}}
        if m == "prompts/get":
            return {"jsonrpc": "2.0", "id": i,
                    "result": {"messages": self.prompts[p["name"]](**p.get("arguments", {}))}}
        return {"jsonrpc": "2.0", "id": i, "error": {"code": -32601, "message": f"method not found: {m}"}}

class MCPClient:
    def __init__(self, server): self.s = server; self._i = 0
    def _c(self, method, params=None):
        self._i += 1; r = self.s.handle(rpc(self._i, method, params))
        if "error" in r: raise RuntimeError(r["error"]["message"])
        return r["result"]
    def list_tools(self): return self._c("tools/list")["tools"]
    def call_tool(self, name, **a): return json.loads(self._c("tools/call",
                                    {"name": name, "arguments": a})["content"][0]["text"])
    def read_resource(self, uri): return json.loads(self._c("resources/read",
                                    {"uri": uri})["contents"][0]["text"])
    def get_prompt(self, name, **a): return self._c("prompts/get", {"name": name, "arguments": a})["messages"]

# Build a support-ops server with one of each primitive
srv = MCPServer()
srv.tools["get_csat"] = {"d": "CSAT for a support channel.",
                         "schema": {"type": "object",
                                    "properties": {"channel": {"type": "string"}},
                                    "required": ["channel"]},
                         "fn": lambda channel: {"channel": channel, "csat": CSAT.get(channel.lower())}}
srv.resources["data://channels"] = lambda: list(CSAT)
srv.prompts["triage"] = lambda channel: [{"role": "user",
    "content": {"type": "text", "text": f"Summarise {channel} issues."}}]

client = MCPClient(srv)
print("tools    :", client.list_tools())
print("tool call:", client.call_tool("get_csat", channel="phone"))
print("resource :", client.read_resource("data://channels"))
print("prompt   :", client.get_prompt("triage", channel="email"))

## 4. An agent that discovers its tools over MCP

The payoff: the agent doesn't hard-code tools — it reads them from the server via `list_tools()` and calls them with `call_tool()`. The same agent works against *any* MCP server.

In [ ]:
def mcp_agent(question, client):
    tools = {t["name"] for t in client.list_tools()}      # discovered at runtime
    q = question.lower()
    if "get_csat" in tools:
        ch = next((c for c in CSAT if c in q), "chat")
        return client.call_tool("get_csat", channel=ch)
    return "no suitable tool"

print(mcp_agent("what is the csat for social?", client))

**The real thing:** in production you use the `mcp` SDK — `FastMCP` turns plain functions into a server with `@mcp.tool()` decorators — and register it with a Claude host:

```bash
claude mcp add support-ops -- python support_ops_server.py     # Claude Code
```

The protocol is exactly what you built above; only the transport (stdio/HTTP) changes.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a tool to the live server

Register a `count_channels` tool, then call it through the client (no client changes — that's MCP's point).

In [ ]:
# Your code here  👇
# Register a `count_channels` tool on `srv` (a dict with "d", optional "schema", and "fn"),
# then call it through `client.call_tool(...)` — no client changes needed.


<details>
<summary>💡 <b>Solution</b></summary>

```python
srv.tools["count_channels"] = {"d": "How many channels exist.",
                               "fn": lambda: {"count": len(CSAT)}}
print(client.call_tool("count_channels"))
```

**Reasoning.** This is MCP's core promise in two lines: the server gains a capability and **no client code changes**. The client never imported `count_channels` — it discovers it through `tools/list` and invokes it by name through `tools/call`. (We omitted `"schema"`, so the server advertises the default empty `inputSchema` — correct for a tool that takes no arguments.) In a real deployment this is exactly how you ship new functionality to every connected host (Claude Desktop, Claude Code, your app) by redeploying only the server.
</details>

### Exercise 2 — ⭐⭐ Validate an enum

Add a `status` field restricted to `["open", "closed"]` and show `validate` rejects `"pending"`.

In [ ]:
# Your code here  👇
S2 = {"type": "object", "properties": {"status": {"type": "string"}}, "required": ["status"]}

def validate_enum(args, allowed=("open", "closed")):
    ...

# print("open   :", validate_enum({"status": "open"}))
# print("pending:", validate_enum({"status": "pending"}))


<details>
<summary>💡 <b>Solution</b></summary>

```python
S2 = {"type": "object", "properties": {"status": {"type": "string"}}, "required": ["status"]}
def validate_enum(args, allowed=("open", "closed")):
    return [] if args.get("status") in allowed else [f"status must be one of {allowed}"]
print("open   :", validate_enum({"status": "open"}))
print("pending:", validate_enum({"status": "pending"}))
```

**Reasoning.** JSON-Schema spells this `"enum": ["open", "closed"]`, and our validator implements the same membership check by hand: return an **empty problem list** on success, a list of human-readable messages on failure — the same contract as the chapter's `validate`. Enums are the cheapest way to stop a model from inventing values (`"pending"`, `"OPEN"`, `"in‐progress"`); the error message even tells the model the legal options, which is exactly what a real LLM needs to self-correct on the next attempt.
</details>

### Exercise 3 — ⭐⭐ Debug me 🐞

This call is meant to fetch CSAT for chat but errors. Read the JSON-RPC error (surfaced as a `RuntimeError`), then fix it (next cell).

In [ ]:
# 🐞 BUG (INTENTIONALLY ERRORS): the tool wants 'channel', not 'chanel'.
print(client.call_tool("get_csat", chanel="chat"))

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# ✅ Fix: use the argument name the tool advertises.
print(client.call_tool("get_csat", channel="chat"))
```

**Reasoning.** The buggy call sent `chanel=` but the tool's `inputSchema` (visible via `client.list_tools()`) advertises `channel`. The server caught the resulting `TypeError` and returned a structured JSON-RPC **error** object (`code -32602`, invalid params) instead of a raw traceback; the client surfaced it as a `RuntimeError`. That's the whole robustness lesson of section 2 applied to the protocol: the model (or you) reads the error message, corrects the argument name, and retries — the server never crashes.
</details>

## 🧠 Stretch exercises

### Stretch exercise C — ⭐⭐⭐ Two servers, one router

Real hosts connect to several servers. Route a tool call to whichever server advertises it.

In [ ]:
# Your code here  👇
math = MCPServer()
math.tools["add"] = {"d": "Add two numbers.", "fn": lambda a, b: {"sum": a + b}}
clients = [client, MCPClient(math)]

def route_call(tool, **args):
    ...

# print(route_call("get_csat", channel="email"))
# print(route_call("add", a=2, b=3))


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
math = MCPServer()
math.tools["add"] = {"d": "Add two numbers.", "fn": lambda a, b: {"sum": a + b}}
clients = [client, MCPClient(math)]

def route_call(tool, **args):
    for c in clients:
        if tool in {t["name"] for t in c.list_tools()}:
            return c.call_tool(tool, **args)
    raise KeyError(tool)

print(route_call("get_csat", channel="email"))
print(route_call("add", a=2, b=3))
```

**Reasoning.** This is exactly what a real MCP **host** does: it keeps one client per server (the 1:1 rule from section 3) and builds a routing table from each server's `tools/list`. Discovery makes the router generic — add a third server to `clients` and its tools become callable with zero routing changes. Two production wrinkles to know about: hosts usually **cache** the tool list instead of re-listing on every call, and they need a policy for **name collisions** (two servers advertising the same tool name) — most prefix tool names with the server name.
</details>

### Stretch exercise D — ⭐⭐⭐ A logging transport

Wrap the client's calls to print every request method — the MCP equivalent of `tcpdump`.

In [ ]:
# Your code here  👇
class LoggingClient(MCPClient):
    def _c(self, method, params=None):
        ...

# dbg = LoggingClient(srv)
# dbg.call_tool("get_csat", channel="phone")


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class LoggingClient(MCPClient):
    def _c(self, method, params=None):
        print("→", method, params or {})
        return super()._c(method, params)

dbg = LoggingClient(srv)
dbg.call_tool("get_csat", channel="phone")
```

**Reasoning.** Because every message funnels through the single `_c` method, subclassing and overriding it gives you a complete wire log in four lines — print the method and params, then delegate with `super()._c(...)`. This is the decorator/middleware pattern from NB 4 applied to a transport, and it's how real MCP debugging works too: log the JSON-RPC traffic, not the Python internals. In production you'd write to a structured logger (with timestamps and request ids) instead of `print`, but the interception point is identical.
</details>

## 🧠 Key takeaways

1. An **agent** = a model deciding the next step in a **bounded** loop; always set `max_steps`.
2. Give every tool a **JSON-Schema**, **validate** arguments, and return a consistent **`{ok, result|error}`** envelope.
3. **MCP** is one open standard for connecting AI apps to **tools / resources / prompts** over **JSON-RPC** — write a server once, use it in Claude Desktop, Claude Code, or your app.
4. An MCP agent **discovers** its tools at runtime, so it's portable across servers.
5. Offline stand-ins and a real model/SDK are interchangeable — the loop and the protocol don't change.

## ✅ Self-assessment

- [ ] Run a ReAct loop with a step budget
- [ ] Validate tool arguments against a JSON-Schema
- [ ] Build a mini MCP server + client and call all three primitives
- [ ] Connect an agent that discovers tools over MCP

## 🚀 Next step

You've finished the fast track! 🎉 For the deep version of this material — planning, reflection, memory, robust tool registries, the full MCP protocol, and a multi-agent capstone — open the full course's **[Module 8](../08_agents_tools_mcp/)**. Or jump back to [`../00_onboarding/00_master_onboarding.ipynb`](../00_onboarding/00_master_onboarding.ipynb) for everything else the fast track trimmed.